# Example 11.6.1 / 1: NCP, MCP, and MPCC

Companion notebook for **11.6.1 Equilibrium Problems and MPCC** in the AMPL lab Documentation, section "NCP, MCP, and MPCC". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## NCP, MCP, and MPCC

| Class | Structure | Typical solver interpretation | Concrete instance |
|---|---|---|---|
| NCP | $0\le x\perp F(x)\ge0$ | Find a root satisfying nonnegativity and complementarity | the `Market` pair $0\le p\perp 2p-6\ge0$ in Complementarity Constraints (AMPL lab documentation, section 11.6): $p=3$ |
| MCP | Variables may have lower and upper bounds with associated residual signs | Solve a mixed complementarity system | the `BoundedPair` model $0\le x\le10\perp x-c$ in Complementarity Constraints (AMPL lab documentation, section 11.6): $x=0$, $4$ or $10$ |
| MPCC | Optimize an objective subject to complementarity and other constraints | Nonlinear optimization with difficult constraint geometry | the bilevel `Upper` model below: objective 2 at $x=y=0$ |

These names are not interchangeable. PATH is designed for square complementarity systems; an MPCC needs an optimization solver and appropriate handling of complementarity. A square count is necessary for the standard MCP interface but is not an existence or uniqueness theorem: degeneracy can permit multiple equilibria, while inconsistent relations can permit none.

A square count does not settle existence or uniqueness. For the one-variable pair
$0\le x\perp ax-b\ge0$, three data sets give a unique solution, a continuum, and none:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
param a;  # slope of the residual
param b;  # offset of the residual

var x;  # sign set by the pair

subject to Pair: x >= 0 complements a * x - b >= 0;  # x > 0 only where a * x - b is zero

Writing model.mod


### amplpy: run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# run.py
for a, b in ((2, 6), (0, 0), (0, 1)):
    ampl.param["a"] = a
    ampl.param["b"] = b
    ampl.var["x"].set_value(0)
    ampl.solve(solver="highs", verbose=False)
    print((a, b), ampl.solve_result, round(ampl.var["x"].value(), 6))

(2, 6) solved 3.0
(0, 0) solved 0.0
(0, 1) infeasible 0.0
